In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, save_npz
import joblib

In [8]:
df = pd.read_csv("preprocessedDataFrame.csv")
df.drop(columns='Review' , inplace=True)
print(f'empty entries = {df["processedReview"].isna().sum()}')
df = df.dropna(subset=["processedReview"])

empty entries = 2


**Note:**  
I can't use all unique words for `max_features` because the number is too large (over 2 million).


In [9]:
# Combine all words into one long string and split
all_words = ' '.join(df['processedReview']).split()

# Count unique terms
vocab_size = len(set(all_words))

print(f"Total number of unique words in the dataset: {vocab_size}") 

Total number of unique words in the dataset: 2165246


In [10]:
text = df["processedReview"].tolist()
tfidf = TfidfVectorizer(
    max_features=10000,
    min_df=6,             # filter out extremely rare terms
    max_df=0.9,           # filter out very frequent terms
    ngram_range=(1,1)     # just unigrams, since stop words are already removed
)
tfidf_matrix = tfidf.fit_transform(text)

# Save vectorizer for later
joblib.dump(tfidf, "tfidf_vectorizer.joblib")

features_df = df[['Rating', 'failedNumberConversion', 'sentenceLength', 'numExclamations']].reset_index(drop=True)

# Convert features_df to sparse matrix and align types
features_sparse = pd.DataFrame(features_df).astype(float).values

# Combine sparse TF-IDF matrix with numerical features
combined_sparse = hstack([tfidf_matrix, features_sparse])

# Convert to DataFrame (dense at this step — ensure it's manageable or save directly)
column_names = list(tfidf.get_feature_names_out()) + list(features_df.columns)

In [11]:
# Combine
combined_sparse = hstack([tfidf_matrix, features_sparse])

# Save sparse matrix
save_npz("tfidfPlusFeatures.npz", combined_sparse)

# Save column names separately
column_names = list(tfidf.get_feature_names_out()) + list(features_df.columns)
joblib.dump(column_names, "tfidfPlusFeaturesColumns.pkl")

print("Sparse matrix and column names saved successfully.")

Sparse matrix and column names saved successfully.


<h5>Fares Alfares</h5>